# 01 — Analyse Exploratoire des Données (EDA)
**Projet** : ObRail MSPR 2025-2026  
**Auteure** : Charlotte  
**Source** : `data/processed/routes_processed.csv`  
**Objectif** : Comprendre la structure du dataset, la distribution des variables et les relations entre elles. Aucune transformation n'est appliquée ici.

## 0. Imports et configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for folder in [cwd] + list(cwd.parents):
        if (folder / 'data').exists() and (folder / 'src').exists() and (folder / 'models').exists():
            return folder
    raise FileNotFoundError('Could not find project root from current working directory')

ROOT = find_project_root()
DATA_PATH = ROOT / 'data' / 'processed' / 'routes_processed.csv'
PLOT_DIR = ROOT / 'evaluation' / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'is_underserved'

print('✅ Imports OK')

## 1. Structure du dataset
On commence par comprendre ce qu'on a : dimensions, types, valeurs manquantes.

In [ ]:
df = pd.read_csv(DATA_PATH, dtype={'days_of_week': str})
print(f'Shape : {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')

In [ ]:
df.info()

In [ ]:
df.describe().T

### 1.1 Valeurs manquantes

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0]

print('Colonnes avec valeurs manquantes :')
display(missing_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(missing_df.index, missing_df['pct'], color='#c0392b')
ax.set_xlabel('% manquant')
ax.set_title('Taux de valeurs manquantes par colonne')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'missing_values.png')
plt.show()

# Note : route_short_name et route_long_name sont des métadonnées textuelles.
# Elles ne seront pas utilisées comme features — leurs valeurs manquantes
# n'ont pas d'impact sur la modélisation.

## 2. Distribution de la cible `is_underserved`
Avant d'explorer les features, on observe la variable qu'on cherche à prédire.

In [ ]:
counts = df[TARGET].value_counts()
ratio  = counts[0] / counts[1]

print('Distribution is_underserved :')
print(counts)
print(f'\nRatio majoritaire / minoritaire : {ratio:.2f}:1')
print(f'Taux de sous-desserte : {counts[1]/len(df)*100:.1f} %')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Non sous-desservi (0)', 'Sous-desservi (1)'],
            counts.values, color=['#2980b9', '#e74c3c'])
axes[0].set_title('Effectifs par classe')
axes[0].set_ylabel('Nombre de routes')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values,
            labels=['Non sous-desservi', 'Sous-desservi'],
            colors=['#2980b9', '#e74c3c'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Répartition proportionnelle')

plt.suptitle('Déséquilibre de classes — is_underserved', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'class_balance.png')
plt.show()

# Observation : le dataset est déséquilibré (~4:1).
# Conséquence directe : l'accuracy seule sera une métrique trompeuse
# (un modèle qui prédit toujours 0 obtiendrait ~80% d'accuracy).
# Ce déséquilibre devra être pris en compte lors de la modélisation.

## 3. Distribution des variables numériques
On observe la forme de chaque distribution : asymétrie, outliers, plages de valeurs.

In [ ]:
num_cols = ['distance_km', 'log_distance', 'co2_per_pkm',
            'log_co2', 'rail_modal_share', 'days_active']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40,
                 color='#3498db', edgecolor='white', alpha=0.8)
    skew = df[col].dropna().skew()
    axes[i].set_title(col)
    axes[i].text(0.97, 0.95, f'skew={skew:.2f}',
                 transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=9)

plt.suptitle('Distribution des variables numériques', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'num_distributions.png')
plt.show()

### 3.1 Distributions numériques par classe cible
Est-ce que les features numériques séparent visuellement les deux classes ?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for cls, color, label in [
        (0, '#2980b9', 'Non sous-desservi'),
        (1, '#e74c3c', 'Sous-desservi')
    ]:
        subset = df[df[TARGET] == cls][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.5,
                     color=color, label=label, density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Distributions par classe cible (densité normalisée)', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'num_by_class.png')
plt.show()

# Observation : si les deux histogrammes se superposent complètement
# pour une feature, celle-ci n'apporte pas de signal discriminant.

### 3.2 Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['distance_km', 'co2_per_pkm', 'rail_modal_share']):
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('is_underserved')

plt.suptitle('Boxplots par classe cible')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'boxplots_by_class.png')
plt.show()

In [ ]:
# Quantification des outliers sur distance_km (méthode IQR)
Q1  = df['distance_km'].quantile(0.25)
Q3  = df['distance_km'].quantile(0.75)
IQR = Q3 - Q1

outliers = df[
    (df['distance_km'] < Q1 - 1.5 * IQR) |
    (df['distance_km'] > Q3 + 1.5 * IQR)
]

print(f'Outliers distance_km : {len(outliers):,} ({len(outliers)/len(df)*100:.1f}%)')
print(f'  Min    : {df["distance_km"].min():.1f} km')
print(f'  Médiane: {df["distance_km"].median():.1f} km')
print(f'  Max    : {df["distance_km"].max():.1f} km')

## 4. Distribution des variables catégorielles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['service_type', 'is_international']):
    vc = df[col].value_counts()
    ax.bar(vc.index.astype(str), vc.values, color='#3498db')
    ax.set_title(f'Distribution — {col}')
    ax.set_ylabel('Effectif')
    for i, v in enumerate(vc.values):
        ax.text(i, v + 50, str(v), ha='center')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'cat_distributions.png')
plt.show()

In [ ]:
# Répartition par pays de départ — top 20
top20 = df['departure_country'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(top20.index, top20.values, color='#3498db')
ax.set_title('Nombre de routes par pays de départ (top 20)')
ax.set_ylabel('Effectif')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'routes_by_country.png')
plt.show()

## 5. Relations entre variables et cible
On cherche à observer quelles variables sont associées à la sous-desserte.

### 5.1 Taux de sous-desserte par type de service

In [ ]:
rate_by_type = (
    df.groupby('service_type')[TARGET]
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'underserved', 'count': 'total', 'mean': 'rate'})
)
rate_by_type['rate_pct'] = (rate_by_type['rate'] * 100).round(1)
display(rate_by_type)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(rate_by_type.index, rate_by_type['rate_pct'],
              color=['#3498db', '#e74c3c'])
ax.set_ylabel('% sous-desservi')
ax.set_ylim(0, 100)
ax.set_title('Taux de sous-desserte par type de service')
for bar, val in zip(bars, rate_by_type['rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1, f'{val}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'underserved_by_service_type.png')
plt.show()

# Observation : les trains de nuit circulent structurellement moins de jours
# par semaine. Leur taux élevé de sous-desserte reflète une réalité opérationnelle,
# pas un problème de données.

### 5.2 Taux de sous-desserte par pays de départ (top 15)

In [ ]:
rate_by_country = (
    df.groupby('departure_country')[TARGET]
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'underserved', 'count': 'total', 'mean': 'rate'})
    .sort_values('total', ascending=False)
    .head(15)
)
rate_by_country['rate_pct'] = (rate_by_country['rate'] * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if r > 20 else '#3498db'
          for r in rate_by_country['rate_pct']]
ax.bar(rate_by_country.index, rate_by_country['rate_pct'], color=colors)
ax.axhline(df[TARGET].mean() * 100, color='black',
           linestyle='--', label=f'Moyenne globale ({df[TARGET].mean()*100:.1f}%)')
ax.set_ylabel('% sous-desservi')
ax.set_title('Taux de sous-desserte par pays de départ (top 15 par volume)')
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'underserved_by_country.png')
plt.show()

display(rate_by_country)

### 5.3 Routes domestiques vs internationales

In [ ]:
intl = df.groupby('is_international')[TARGET].agg(['mean', 'count'])
intl.index = ['Domestique', 'International']
intl['rate_pct'] = (intl['mean'] * 100).round(1)
display(intl)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(intl.index, intl['count'], color=['#3498db', '#e67e22'])
axes[0].set_title('Volume de routes')
axes[0].set_ylabel('Effectif')

axes[1].bar(intl.index, intl['rate_pct'], color=['#3498db', '#e67e22'])
axes[1].set_title('Taux de sous-desserte (%)')
axes[1].set_ylabel('%')

plt.suptitle('Routes domestiques vs internationales', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'domestic_vs_international.png')
plt.show()

## 6. Corrélations
On mesure les relations linéaires entre features numériques et avec la cible.
Une corrélation faible ne signifie pas qu'une feature est inutile — elle peut
avoir une relation non-linéaire que les modèles à base d'arbres exploiteront.

In [ ]:
corr_cols = [
    'rail_modal_share', 'type_encoded', 'is_international',
    'log_distance', 'log_co2', 'country_encoded',
    'days_active', 'distance_km', TARGET
]

corr_with_target = (
    df[corr_cols].corr()[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v > 0 else '#2980b9' for v in corr_with_target]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Corrélation de Pearson avec is_underserved')
ax.set_xlabel('Corrélation')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'correlation_target.png')
plt.show()

print('Valeurs :')
print(corr_with_target.round(4))

In [ ]:
# Heatmap des corrélations inter-features
feat_cols = [
    'rail_modal_share', 'type_encoded', 'is_international',
    'log_distance', 'log_co2', 'country_encoded', TARGET
]

fig, ax = plt.subplots(figsize=(9, 7))
corr_matrix = df[feat_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Matrice de corrélation (features + cible)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'correlation_matrix.png')
plt.show()

## 7. Conclusions

**Structure**
- 25 200 routes propres, 34 pays, 21 colonnes
- `route_short_name` et `route_long_name` ont des valeurs manquantes mais ne seront pas utilisées comme features

**Cible**
- Déséquilibre 4.16:1 (80.6% non sous-desservi, 19.4% sous-desservi)
- L'accuracy seule est une métrique inadaptée — un modèle prédisant toujours 0 obtiendrait 80.6% d'accuracy sans rien apprendre

**Variables — signaux forts**
- `service_type` est la variable la plus discriminante : les trains de nuit sont sous-desservis à 75.4% contre 17.9% pour les trains de jour
- `is_international` est un signal très fort : 94.4% des routes internationales sont sous-desservies contre 18.5% des routes domestiques
- Par pays, les écarts sont importants : NL (37.6%), IE (27.7%), DE (22.5%), FR (22.3%) sont au-dessus de la moyenne globale (19.4%), tandis que PT (4.1%), RO (4.8%) et SI (1.7%) sont très en dessous

**Corrélations avec la cible**
- `days_active` (-0.57) et `log_distance` (0.48) ont les corrélations les plus fortes — mais toutes deux sont directement liées à la règle de construction de `is_underserved`
- `type_encoded` (0.23) et `is_international` (0.20) montrent un signal modéré et légitime
- `rail_modal_share` (0.09), `country_encoded` (-0.01) et `log_co2` (-0.01) ont des corrélations linéaires faibles — cela ne les exclut pas, les modèles à base d'arbres peuvent exploiter des relations non-linéaires

**Points à traiter dans `02_feature_engineering.ipynb`**
- `days_active` et `distance_km` sont les sources directes de `is_underserved` — les exclure du feature set
- `log_distance` est corrélée à 0.48 avec la cible via la règle `distance_km > 100` — à surveiller lors de l'analyse de feature importance
- Le déséquilibre de classes (4.16:1) devra être adressé lors de la modélisation (`03_models.ipynb`)